# Entity Ontology Framework — walkthrough

Reproduces Fig. 2 of the paper using Wikidata-derived classes.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.config import load_config
from src.ontology.eob_schema import EoBSchema
from src.entity_linking.kb_cache import KBCache
from src.ontology.build_schema import populate_from_wikidata

cfg = load_config('../configs/default.yaml')
cache = KBCache(cfg.paths.kb_cache)
schema, crossview = populate_from_wikidata(cache, top_level_types=cfg.ontology.top_level_types)
print('top-level dims:', list(schema.root.children))
print('total classes :', len(schema.all_classes()))

In [ ]:
import json
print(json.dumps(schema.to_dict(), indent=2)[:2000])

In [ ]:
import networkx as nx, matplotlib.pyplot as plt
G = nx.DiGraph()
def walk(node, prefix=''):
    for child in node.children.values():
        pid = f'{prefix}{child.name}' if prefix else child.name
        G.add_edge(prefix or 'ROOT', pid)
        walk(child, pid + '.')
walk(schema.root)
fig, ax = plt.subplots(figsize=(10, 6))
pos = nx.spring_layout(G, seed=42, k=0.6)
nx.draw(G, pos, with_labels=True, node_size=900, font_size=7, ax=ax)
ax.set_title('EoBSchema — 4-dim hierarchical entity ontology')
plt.tight_layout()